# Cleaning for `trips.csv`

**5 Cleaning Tasks**

1. **Primary-key validation**: confirm `trip_id` uniqueness  
2. **Datetime conversion**: parse the five time columns  
3. **Invalid-value checks**: flag logical and missing-value issues  
4. **Categorical standardisation**: clean free-text categorical fields  
5. **JSON parsing**: extract stop-level metrics from `route_stops_info`


## Setup

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

# CONFIGURATION 
INPUT_PATH  = "<INPUT_DATA_DIR>/trips.csv"           # <- update if your file is elsewhere
OUTPUT_PATH = "<INPUT_DATA_DIR>/trips_cleaned.csv"


In [ ]:
df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"Loaded {len(df):,} rows  ×  {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
df.head(3)


In [ ]:
# data types and non-null counts
df.info(show_counts=True)


---
## 1. Primary-Key Validation

Check for **nulls** and **duplicates** in `trip_id`.


In [ ]:
# 1a  Null check
null_count = df["trip_id"].isna().sum()
print(f"Null trip_id values:      {null_count}")

# 1b  Uniqueness check
n_unique = df["trip_id"].nunique()
n_rows   = len(df)
print(f"Total rows:               {n_rows:,}")
print(f"Unique trip_id values:    {n_unique:,}")
print(f"Duplicated rows:          {n_rows - n_unique:,}")

if n_rows == n_unique and null_count == 0:
    print("\n✓  trip_id is a valid primary key")


In [ ]:
# 1c  Handle duplicates (if any exist)
dup_mask = df["trip_id"].duplicated(keep=False)
if dup_mask.sum() > 0:
    print(f"Investigating {dup_mask.sum()} duplicated rows:")
    display(df[dup_mask].sort_values("trip_id").head(10))

    # Strategy: keep the first occurrence
    before = len(df)
    df = df.drop_duplicates(subset="trip_id", keep="first").reset_index(drop=True)
    print(f"\nDropped {before - len(df)} duplicate rows  →  {len(df):,} rows remain")
else:
    print("No duplicates found — no action needed.")


---
## 2. Convert Time Fields to `datetime`

Columns to convert:  
`started_at` · `ended_at` · `pickup_time` · `dropoff_time` · `departed_at`

Each column is parsed to datetime, then split into separate **date** and **time** columns.  
The original datetime columns are kept temporarily for the illogical-value checks in Section 3,  
then dropped before export.

Values that cannot be converted are changed to `NaT` (not a time), and we report how many.


In [ ]:
TIME_COLS = ["started_at", "ended_at", "pickup_time", "dropoff_time", "departed_at"]

for col in TIME_COLS:
    before_na = df[col].isna().sum()
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True).dt.tz_convert("America/Chicago")
    after_na  = df[col].isna().sum()
    coerced   = after_na - before_na
    print(f"  {col:20s}  coerced to NaT: {coerced:>5,}   total NaT: {after_na:>5,}  "
          f"({after_na/len(df)*100:.1f}%)")

    # Split into separate date and time columns (no timezone offset)
    df[col + "_date"] = df[col].dt.date
    df[col + "_time"] = df[col].dt.strftime("%H:%M:%S").replace("NaT", np.nan)

print("\n✓  Date and time columns created")


In [ ]:
# Verify it worked
verify_cols = []
for col in TIME_COLS:
    verify_cols.extend([col + "_date", col + "_time"])
df[["trip_id"] + verify_cols].dropna(subset=["started_at_date"]).head(3)


---
## 3. Illogical-Value Checks

We run five checks and combine the results into a single `_flag_illogical` column.  
A row is flagged `True` if it fails **at least one** check.

| Check | Rule |
|---|---|
| Time inversion | `started_at > ended_at` |
| Negative positions count | `trip_positions_count < 0` |
| Missing date | `started_at` is `NaT` (trip never started) |
| Missing route | `route_id` is null |
| Missing vehicle | `vehicle_id` is null |


In [ ]:
df["_flag_illogical"] = (
    # Time inversion (uses original datetime columns before they are dropped)
    (df["started_at"].notna() & df["ended_at"].notna() & (df["started_at"] > df["ended_at"]))
    # Negative trip_positions_count
    | (df["trip_positions_count"] < 0)
    # Missing start date
    | df["started_at"].isna()
    # Missing route_id
    | df["route_id"].isna()
    # Missing vehicle_id
    | df["vehicle_id"].isna()
)

# Drop original datetime columns now that checks are done
df = df.drop(columns=TIME_COLS)

flagged = df["_flag_illogical"].sum()
print(f"Flagged rows:  {flagged:,}  ({flagged/len(df)*100:.1f}%)")
print(f"Clean rows:    {len(df) - flagged:,}")


In [ ]:
# Cross-tab: are the flagged rows just the "pending" trips?
print("Flag breakdown by status:")
print(pd.crosstab(df["status"], df["_flag_illogical"], margins=True))


## 3b. Optional drop of 'flagged rows'
Enable the cell below to drop all flagged rows


In [ ]:
# # OPTIONAL: drop all flagged rows
# before = len(df)
# df = df[~df["_flag_illogical"]].reset_index(drop=True)
# print(f"Dropped {before - len(df)} flagged rows  →  {len(df):,} rows remain")


---
## 4. Standardise Categorical Fields

Target columns: `trip_type` · `status` · `vehicle_type` · `source` · `start_trigger` · `end_trigger`

Steps applied to each:
1. Strip whitespace  
2. Convert to lowercase  
3. Collapse spaces/underscores  

Print value counts for manual review.  


In [ ]:
import re

CAT_COLS = ["trip_type", "status", "vehicle_type", "source", "start_trigger", "end_trigger"]

def standardise(series: pd.Series) -> pd.Series:
    """Lowercase, strip, collapse whitespace/underscores."""
    s = series.astype(str).str.strip().str.lower()
    s = s.str.replace(r"[\s_]+", "_", regex=True)
    s = s.replace({"nan": np.nan, "none": np.nan, "": np.nan})
    return s

for col in CAT_COLS:
    df[col] = standardise(df[col])
    vc = df[col].value_counts(dropna=False)
    print(f"── {col} ({vc.shape[0]} unique) ──")
    print(vc.to_string())
    print()


### 4b. Manual Mappings (Optional)

If future loads introduce variants (e.g. `"Bus"` vs `"bus"`, `"cancelled"` vs `"canceled"`),
add explicit mappings below.


In [ ]:
#Add manual corrections here if needed 
# Example:
# df["status"] = df["status"].replace({
#     "canceled": "cancelled",   # normalise spelling variant
# })
#
# df["vehicle_type"] = df["vehicle_type"].replace({
#     "minivan": "van",
# })


---
## 5. Parse `route_stops_info` JSON

### Observed structure

Each cell contains a **JSON object** keyed by `route_stop_id`. Each value is a dict like:
```json
{
  "4743": {
    "mileage": 8.82,
    "eta_diff": 9,
    "eta_time": "2025-09-02T07:29:06.597-05:00",
    "completed": true,
    "completed_diff": 8,
    "completed_time": "2025-09-02T12:28:25.000Z",
    "departed": true,
    "departed_diff": 6,
    "minutes_to_arrival": 0
  },
  ...
}
```

Empty trips have `{}`.

### Two outputs

**Long-format table (`trip_stops_long.csv`)** — one row per trip × stop, preserving per-stop
detail for downstream analysis (stop matching, dwell time, per-stop delay distributions).

| Column | Description |
|---|---|
| `trip_id` | Foreign key back to the trips table |
| `route_stop_id` | The JSON key — references `route_stops.csv` |
| `completed` | Whether the geofence detected arrival (`True`/`False`) |
| `completed_diff` | Minutes from planned arrival (negative = early, positive = late) |
| `completed_time` | Timestamp when geofence completion was detected |
| `departed` | Whether the geofence detected departure (`True`/`False`) |
| `departed_diff` | Minutes from planned departure |
| `eta_diff` | How far off the ETA prediction was (minutes) |
| `eta_time` | The predicted ETA timestamp |
| `mileage` | Distance to this stop (miles) |
| `minutes_to_arrival` | Real-time ETA snapshot at time of detection |

**Wide-format aggregates** — trip-level summary columns added back to `trips_cleaned.csv`.

| Column | Description |
|---|---|
| `rsi_stop_count` | Total stops in the JSON |
| `rsi_completed_count` | Stops where geofence detected arrival |
| `rsi_incomplete_count` | Stops not completed |
| `rsi_departed_count` | Stops where geofence detected departure |
| `rsi_avg_completed_diff` | Mean arrival diff across stops (minutes) |
| `rsi_avg_departed_diff` | Mean departure diff across stops (minutes) |
| `rsi_avg_eta_diff` | Mean ETA prediction error across stops (minutes) |
| `rsi_total_mileage` | Sum of mileage across all stops (miles) |

> **Note on sign convention:** For `completed_diff` and `departed_diff`, negative = early, positive = late.


In [ ]:
def explode_route_stops_info(df):
    """
    Parse route_stops_info JSON from every row and return a long-format
    DataFrame with one row per trip x stop.
    """
    records = []

    for _, row in df.iterrows():
        raw = row["route_stops_info"]
        trip_id = row["trip_id"]

        # Skip empty / missing
        if pd.isna(raw) or raw in ("", "{}", "nan"):
            continue

        try:
            data = json.loads(raw) if isinstance(raw, str) else raw
        except (json.JSONDecodeError, TypeError):
            continue

        if not isinstance(data, dict) or len(data) == 0:
            continue

        for stop_id, stop in data.items():
            if not isinstance(stop, dict):
                continue

            records.append({
                "trip_id": trip_id,
                "route_stop_id": int(stop_id),
                "completed": stop.get("completed"),
                "completed_diff": stop.get("completed_diff"),
                "completed_time": stop.get("completed_time"),
                "departed": stop.get("departed"),
                "departed_diff": stop.get("departed_diff"),
                "eta_diff": stop.get("eta_diff"),
                "eta_time": stop.get("eta_time"),
                "mileage": stop.get("mileage"),
                "minutes_to_arrival": stop.get("minutes_to_arrival"),
            })

    return pd.DataFrame(records)


In [ ]:
# Build the long-format stop table
stops_long = explode_route_stops_info(df)

# Convert types
stops_long["completed"] = stops_long["completed"].fillna(False).astype(bool)
stops_long["departed"]  = stops_long["departed"].fillna(False).astype(bool)

for col in ["completed_diff", "departed_diff", "eta_diff", "mileage", "minutes_to_arrival"]:
    stops_long[col] = pd.to_numeric(stops_long[col], errors="coerce")

for col in ["completed_time", "eta_time"]:
    stops_long[col] = pd.to_datetime(stops_long[col], errors="coerce", utc=True).dt.tz_convert("America/Chicago")

print(f"Long-format stop table: {len(stops_long):,} rows  ×  {stops_long.shape[1]} columns")
print(f"Trips represented: {stops_long['trip_id'].nunique():,}")
print(f"Unique route_stop_ids: {stops_long['route_stop_id'].nunique():,}")


In [ ]:
stops_long.info(show_counts=True)


In [ ]:
# Summary statistics
print("Numeric column summary:\n")
print(stops_long.describe().round(2).to_string())


In [ ]:
# Completion rates
total = len(stops_long)
completed = stops_long["completed"].sum()
departed  = stops_long["departed"].sum()

print(f"Total stop records:    {total:,}")
print(f"Completed (arrived):   {completed:,}  ({completed/total*100:.1f}%)")
print(f"Departed:              {departed:,}  ({departed/total*100:.1f}%)")


In [ ]:
# Peek at the long-format data
display(stops_long.head(10))


### 5b. Aggregate stop metrics back to trip level (wide format)

These summary columns are added to `trips_cleaned.csv` so trip-level analysis
can use them directly without loading the full stop table.


In [ ]:
# Aggregate from stops_long back to one row per trip
rsi_agg = stops_long.groupby("trip_id").agg(
    rsi_stop_count       = ("route_stop_id", "count"),
    rsi_completed_count  = ("completed", "sum"),
    rsi_departed_count   = ("departed", "sum"),
    rsi_avg_completed_diff = ("completed_diff", "mean"),
    rsi_avg_departed_diff  = ("departed_diff", "mean"),
    rsi_avg_eta_diff       = ("eta_diff", "mean"),
    rsi_total_mileage      = ("mileage", "sum"),
).reset_index()

rsi_agg["rsi_incomplete_count"] = rsi_agg["rsi_stop_count"] - rsi_agg["rsi_completed_count"]

# Merge into the trips dataframe
df = df.merge(rsi_agg, on="trip_id", how="left")

# Fill trips with no stop data (empty JSON) with zeros / NaN
for col in ["rsi_stop_count", "rsi_completed_count", "rsi_incomplete_count", "rsi_departed_count"]:
    df[col] = df[col].fillna(0).astype(int)

print("Wide-format RSI columns added to trips_cleaned:\n")
print(df[["rsi_stop_count", "rsi_completed_count", "rsi_incomplete_count",
          "rsi_departed_count", "rsi_avg_completed_diff", "rsi_avg_departed_diff",
          "rsi_avg_eta_diff", "rsi_total_mileage"]].describe().round(2).to_string())


---
## 6. Final Summary & Export


In [ ]:
# Drop the raw JSON column now that it has been parsed
df = df.drop(columns=["route_stops_info"])

print(f"trips_cleaned:  {df.shape[0]:,} rows  ×  {df.shape[1]} columns")
print(f"stops_long:     {stops_long.shape[0]:,} rows  ×  {stops_long.shape[1]} columns")


In [ ]:
# Missing-value report for all columns
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pct = (missing / len(df) * 100).round(1)
print("Columns with missing values:\n")
print(pd.DataFrame({"missing_count": missing, "pct_missing": pct}).to_string())


In [ ]:
# Final dtypes
print("Column dtypes after cleaning:\n")
print(df.dtypes.to_string())


In [ ]:
# Save cleaned files
df.to_csv(OUTPUT_PATH, index=False)
print(f"✓  Cleaned trips saved → {OUTPUT_PATH}")
print(f"   {df.shape[0]:,} rows  ×  {df.shape[1]} columns")

stops_output = OUTPUT_PATH.replace("trips_cleaned", "trip_stops_long")
stops_long.to_csv(stops_output, index=False)
print(f"\n✓  Stop-level data saved → {stops_output}")
print(f"   {stops_long.shape[0]:,} rows  ×  {stops_long.shape[1]} columns")
